In [1]:
%pip install scikit-learn

  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/72/8d/27c054166bac671770d1ea0ef7716134fe8119c0df224a58c89fd735a61c/scikit_learn-1.9.1-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for scipy>=1.10.0 from https://files.pythonhosted.org/packages/39/e7/979fd14e75008623df31ba70d6bb144700f68feadcea042021c06a05bf82/scipy-1.18.1-cp312-cp312-win_amd64.whl.metadata
  Using cached scipy-1.18.1-cp312-cp312-win_amd64.whl.metadata (61 kB)
  Obtaining dependency information for joblib>=1.4.0 from https://files.pythonhosted.org/packages/18/53/84099323c2ec4be98d935f63c033ac4151ee83836ca1050ede3b3aadf155/joblib-1.6.0-py3-none-any.whl.metadata
  Obtaining dependency information for narwhals>=2.0.1 from https://files.pythonhosted.org/packages/cf/82/8ef9c7e0f50493307a49c09cfa7f00c5a778b6d1921f7b2dabe1d3a5809e/narwhals-2.28.0-py3-none-any.whl.metadata
  Obtaining dependency information for threadpoolctl>=3.5.0 from https://files.python


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Workshop 3 – Entrenamiento del modelo

Entrada: `data/processed/happiness_unified.csv` (generado por `etl/transform.py`).
Salida: `models/model.pkl` (preprocesamiento + modelo).

In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path('../data/processed/happiness_unified.csv')
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(781, 9)


,country,year,score,gdp,family,health,freedom,generosity,corruption
0,Afghanistan,2015,3.575,0.31982,0.30285,0.30335,0.23414,0.36510,0.09719
1,Albania,2015,4.959,0.87867,0.80434,0.81325,0.35733,0.14272,0.06413
2,Algeria,2015,5.605,0.93929,1.07772,0.61766,0.28579,0.07822,0.17383
3,Angola,2015,4.033,0.75778,0.86040,0.16683,0.10384,0.12344,0.07122
4,Argentina,2015,6.574,1.05351,1.24823,0.78723,0.44974,0.11451,0.08484


## Features y variable objetivo

Se usan las seis variables del contrato de eventos. No se usan `country` ni `year`, y se excluyen columnas derivadas del puntaje (`Dystopia Residual`, rangos) para evitar fuga de información.

In [2]:
from sklearn.model_selection import train_test_split

FEATURES = ['gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']
TARGET = 'score'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)
print('Features:', list(X_train.columns))

Entrenamiento: (546, 6) | Prueba: (235, 6)
Features: ['gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']


## Pipeline y entrenamiento

El pipeline une el escalado (`StandardScaler`) y la regresión lineal en un solo objeto. Así el consumidor de Kafka aplicará exactamente el mismo preprocesamiento al predecir.

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression()),
])

model.fit(X_train, y_train)
print('Modelo entrenado')

Modelo entrenado


In [4]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'MAE : {mae:.3f}')
print(f'RMSE: {rmse:.3f}')
print(f'R2  : {r2:.3f}')

MAE : 0.418
RMSE: 0.546
R2  : 0.766


In [5]:
import pandas as pd

coef = pd.Series(model.named_steps['regressor'].coef_, index=FEATURES).sort_values(ascending=False).round(3)
coef

gdp           0.460
family        0.241
health        0.228
freedom       0.195
corruption    0.133
generosity    0.066
dtype: float64

## Guardar el modelo

Se serializa el pipeline completo (escalado + regresión) como `models/model.pkl`. El consumidor de Kafka lo cargará para predecir.

In [6]:
import joblib

MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / 'model.pkl'

joblib.dump(model, MODEL_PATH)
print('Guardado en:', MODEL_PATH.resolve())

# Verificación: cargar el archivo y comprobar que predice igual
loaded = joblib.load(MODEL_PATH)

ejemplo = pd.DataFrame([{
    'gdp': 1.2, 'family': 0.8, 'health': 0.9,
    'freedom': 0.6, 'generosity': 0.3, 'corruption': 0.1,
}])[FEATURES]

print('Predicción de ejemplo:', round(float(loaded.predict(ejemplo)[0]), 3))
print('¿Mismas predicciones que el modelo original?', bool((loaded.predict(X_test) == model.predict(X_test)).all()))

Guardado en: C:\Users\Windows 11\OneDrive\Escritorio\Universidad\Semestre 5\ETL\project\models\model.pkl
Predicción de ejemplo: 6.001
¿Mismas predicciones que el modelo original? True


In [7]:
scaler = model.named_steps['scaler']
reg = model.named_steps['regressor']

print('Medias :', dict(zip(FEATURES, scaler.mean_.round(3))))
print('Desv.  :', dict(zip(FEATURES, scaler.scale_.round(3))))
print('Pesos  :', dict(zip(FEATURES, reg.coef_.round(3))))
print('Intercepto:', round(float(reg.intercept_), 3))

# Predicción manual con el ejemplo del taller
x = ejemplo.values[0]
z = (x - scaler.mean_) / scaler.scale_
manual = reg.intercept_ + (z * reg.coef_).sum()

print('A mano       :', round(float(manual), 3))
print('model.predict:', round(float(model.predict(ejemplo)[0]), 3))

Medias : {'gdp': np.float64(0.915), 'family': np.float64(1.08), 'health': np.float64(0.61), 'freedom': np.float64(0.407), 'generosity': np.float64(0.216), 'corruption': np.float64(0.124)}
Desv.  : {'gdp': np.float64(0.4), 'family': np.float64(0.328), 'health': np.float64(0.254), 'freedom': np.float64(0.149), 'generosity': np.float64(0.123), 'corruption': np.float64(0.105)}
Pesos  : {'gdp': np.float64(0.46), 'family': np.float64(0.241), 'health': np.float64(0.228), 'freedom': np.float64(0.195), 'generosity': np.float64(0.066), 'corruption': np.float64(0.133)}
Intercepto: 5.352
A mano       : 6.001
model.predict: 6.001
